# Experimentos de generalización YOLO11 (plantilla, NO ejecutada)

Los datasets `train_nitido.zip` y `split_por_escenas.zip` se suben a Google Drive y se descomprimen desde este notebook. **No hay credenciales en el notebook**: no se descarga nada de Roboflow.

Reglas: **3 semillas** por configuración (0/1/2), media ± desviación, selección siempre por **valid**, **test una sola vez** al final con el modelo ganador, latencia p95 < 100 ms.

Escenarios: **E0** baseline v1, **E1** `train_nitido`, **E2** versión 2 sin augmentations de Roboflow y Resize Fit con bordes negros (dataset a preparar fuera de este notebook), **E3** split por escenas sin fuga.

`best.pt` y el mAP@50 oficial 0,562 no se modifican.

In [ ]:
import os, glob, json, time, shutil, zipfile
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from ultralytics import YOLO

try:
    from google.colab import drive
    drive.mount('/content/drive')
    EN_COLAB = True
except Exception:
    EN_COLAB = False
print('Google Drive montado:', EN_COLAB)

# Rutas configurables: cambia aquí o define variables de entorno.
DRIVE_RAIZ = os.environ.get('ECOLEARN_DRIVE_RAIZ', '/content/drive/MyDrive/EcoLearn')
SALIDA = os.environ.get('ECOLEARN_SALIDA', f'{DRIVE_RAIZ}/resultados')
ZIP_NITIDO = os.environ.get('ECOLEARN_ZIP_NITIDO', f'{DRIVE_RAIZ}/train_nitido.zip')
ZIP_ESCENAS = os.environ.get('ECOLEARN_ZIP_ESCENAS', f'{DRIVE_RAIZ}/split_por_escenas.zip')
os.makedirs(SALIDA, exist_ok=True)

def descomprimir_si_falta(zip_path, destino):
    if (Path(destino) / 'data.yaml').exists():
        print('Ya extraído:', destino)
        return
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(destino)
    print('Extraído:', destino, '<-', zip_path)

In [ ]:
descomprimir_si_falta(ZIP_NITIDO, DRIVE_RAIZ)
descomprimir_si_falta(ZIP_ESCENAS, DRIVE_RAIZ)

DATA_NITIDO = os.environ.get('ECOLEARN_NITIDO', f'{DRIVE_RAIZ}/train_nitido')
DATA_ESCUENAS = os.environ.get('ECOLEARN_ESCENAS', f'{DRIVE_RAIZ}/split_por_escenas')
DATASET = os.environ.get('ECOLEARN_DATASET', f'{DRIVE_RAIZ}/Capstone-1')
DATA_V2 = os.environ.get('ECOLEARN_V2', f'{DRIVE_RAIZ}/Capstone-2-fit')

PESO_BASE = 'yolo11n.pt'
SEMILLAS = [0, 1, 2]
BASE = dict(epochs=50, imgsz=640, batch=16, patience=15)

EXPERIMENTOS = [
    dict(nombre='E0_baseline_v1', data=f'{DATASET}/data.yaml',
         descripcion='v1 tal cual: Stretch 640, 4 copias, augmentations de Roboflow'),
    dict(nombre='E1_train_nitido', data=f'{DATA_NITIDO}/data.yaml',
         descripcion='solo la copia mas nitida de cada original; valid/test intactos'),
    dict(nombre='E2_v2_fit_sin_aug', data=f'{DATA_V2}/data.yaml',
         descripcion='sin augmentations de Roboflow, Resize Fit con bordes negros'),
    dict(nombre='E3_split_escenas', data=f'{DATA_ESCUENAS}/data.yaml',
         descripcion='307 originales, reparto por rafaga sin fuga'),
]
for e in EXPERIMENTOS:
    print(e['nombre'], '->', e['data'], 'existe:', Path(e['data']).exists())

In [ ]:
def latencia_ms(modelo, imagenes, imgsz=640, repeticiones=3):
    valores = []
    for _ in range(repeticiones):
        for p in imagenes[:30]:
            t0 = time.perf_counter()
            modelo.predict(source=p, imgsz=imgsz, verbose=False)
            valores.append((time.perf_counter() - t0) * 1000)
    return float(np.mean(valores)), float(np.percentile(valores, 95))

def correr(exp, semilla):
    np.random.seed(semilla)
    modelo = YOLO(PESO_BASE)
    nombre_run = f"{exp['nombre']}_s{semilla}"
    modelo.train(data=exp['data'], seed=semilla, project='runs/ecolearn',
                 name=nombre_run, exist_ok=True, **BASE)
    carpeta = Path('runs/ecolearn') / nombre_run
    mejor = YOLO(str(carpeta / 'weights' / 'best.pt'))
    val = mejor.val(data=exp['data'], split='val', verbose=False, plots=False)
    imagenes_val = sorted(glob.glob(f"{os.path.dirname(exp['data'])}/valid/images/*.*"))
    media, p95 = latencia_ms(mejor, imagenes_val, BASE['imgsz'])

    # Artefactos por corrida, guardados en Drive.
    destino = Path(SALIDA) / nombre_run
    destino.mkdir(parents=True, exist_ok=True)
    for rel, nombre in (('results.csv', 'results.csv'),
                        ('weights/best.pt', 'best.pt')):
        origen = carpeta / rel
        if origen.exists():
            shutil.copy2(origen, destino / nombre)
    return dict(nombre=exp['nombre'], semilla=semilla,
                map50_val=float(val.box.map50),
                recall_val=float(val.box.mr),
                latencia_media_ms=media, latencia_p95_ms=p95,
                artefactos=str(destino))

In [ ]:
EJECUTAR = False  # cambiar a True solo cuando se quiera entrenar
resultados = []
if EJECUTAR:
    for exp in EXPERIMENTOS:
        if not Path(exp['data']).exists():
            print('OMITIDA (falta dataset):', exp['nombre'])
            continue
        for semilla in SEMILLAS:
            resultados.append(correr(exp, semilla))
            print('OK', exp['nombre'], semilla, flush=True)
else:
    print('Plantilla cargada. No se entrenó nada.')

## Decisión

Se elige por **valid** (media de 3 semillas, latencia p95 < 100 ms). Solo después se evalúa **test una sola vez** con el modelo ganador. E3 usa un split distinto: sus cifras no son directamente comparables con E0–E2.

In [ ]:
if resultados:
    df = pd.DataFrame(resultados)
    df.to_csv(f'{SALIDA}/resumen_valid.csv', index=False)
    resumen = df.groupby('nombre').agg(
        map50_media=('map50_val', 'mean'), map50_sd=('map50_val', 'std'),
        recall_media=('recall_val', 'mean'),
        latencia_media=('latencia_media_ms', 'mean'),
        latencia_p95=('latencia_p95_ms', 'max')).reset_index()
    display(resumen)
    elegible = resumen[resumen['latencia_p95'] < 100]
    ganador = elegible.loc[elegible['map50_media'].idxmax()]
    varianza = float(resumen['map50_sd'].max())
    print('Mejor por valid (latencia <100 ms):', ganador['nombre'])
    print('¿Supera la varianza entre semillas?',
          bool(ganador['map50_media'] - resumen['map50_media'].min() > varianza))

    fila = (df[df['nombre'] == ganador['nombre']]
            .sort_values('map50_val', ascending=False).iloc[0])
    nombre_run = f"{ganador['nombre']}_s{int(fila['semilla'])}"
    pesos = Path('runs/ecolearn') / nombre_run / 'weights' / 'best.pt'
    mejor = YOLO(str(pesos))
    exp = next(e for e in EXPERIMENTOS if e['nombre'] == ganador['nombre'])
    test = mejor.val(data=exp['data'], split='test', verbose=False, plots=False)
    salida_test = dict(experimento=ganador['nombre'], semilla=int(fila['semilla']),
                       map50_test=float(test.box.map50),
                       fecha=datetime.now().isoformat(timespec='seconds'))
    with open(f'{SALIDA}/resumen_test.json', 'w', encoding='utf-8') as f:
        json.dump(salida_test, f, ensure_ascii=False, indent=2)
    print('TEST (una sola vez) mAP@50 =', round(salida_test['map50_test'], 4))
    print('No repetir la evaluación en test tras ver este resultado.')
else:
    print('Sin resultados todavía.')